# RAG Pipeline: Unstructured Document Processing

**Full End-to-End RAG Pipeline with Unstructured Library**

This notebook demonstrates:
1. Document loading with Unstructured (PDF, DOCX, HTML, etc.)
2. Intelligent chunking respecting content boundaries
3. Embedding generation (sentence-transformers)
4. Vector storage (pgvector)
5. Semantic retrieval
6. LLM response generation

**Setup Requirements:**
```bash
pip install 'unstructured[all-docs]' sentence-transformers sqlalchemy pgvector psycopg2-binary
```

## Step 1: Setup & Imports

In [1]:
import os
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional
from datetime import datetime
import json
import uuid
import warnings
warnings.filterwarnings('ignore')

print("✓ Core imports successful")

✓ Core imports successful


In [2]:
# Load environment
try:
    from dotenv import load_dotenv
    load_dotenv('.env')
    print("✓ .env loaded")
except:
    print("⚠ .env not found")

# Configuration
DATABASE_URL = os.getenv('DATABASE_URL')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
UNSTRUCTURED_API_KEY = os.getenv('UNSTRUCTURED_API_KEY')

print(f"\nConfiguration:")
print(f"  Database: {'✓' if DATABASE_URL else '⚠'}")
print(f"  OpenRouter API: {'✓' if OPENROUTER_API_KEY else '⚠'}")
print(f"  Unstructured API: {'✓' if UNSTRUCTURED_API_KEY else '⚠ (will use local processing)'}")

✓ .env loaded

Configuration:
  Database: ✓
  OpenRouter API: ⚠
  Unstructured API: ✓


In [3]:
# Import LangChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("✓ LangChain imported")

# Import embeddings
from sentence_transformers import SentenceTransformer
import numpy as np
print("✓ Embeddings library imported")

✓ LangChain imported
✓ Embeddings library imported


In [4]:
# Import Unstructured (minimal, avoids matplotlib issues)
try:
    from unstructured.partition.auto import partition
    print("✓ Unstructured partition imported (API or local mode will be used)")
except ImportError as e:
    print(f"❌ Error: {e}")
    print("\nFix: Install with:")
    print("  pip install 'unstructured[all-docs]' --upgrade")
    raise

✓ Unstructured partition imported (API or local mode will be used)


## Step 2: Document Loading (Unstructured)

In [5]:
class SimpleDocumentLoader:
    """Load documents using Unstructured."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key
        self.use_api = bool(api_key)
    
    def load(self, file_path: str) -> Dict[str, Any]:
        """Load document and extract elements."""
        file_path = Path(file_path)
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        
        print(f"Loading: {file_path.name}...")
        
        try:
            # Use Unstructured API if key is available
            if self.use_api:
                print("  Using Unstructured API...")
                elements = partition(
                    file_path=str(file_path),
                    api_key=self.api_key,
                    api_url="https://api.unstructuredapp.io/general/v0/general"
                )
            else:
                print("  Using local processing...")
                elements = partition(str(file_path), infer_table_structure=True)
        except Exception as e:
            print(f"Error: {e}")
            print("\nNote: Install with: pip install 'unstructured[all-docs]'")
            raise
        
        # Classify elements
        result = {
            'file_path': str(file_path),
            'file_name': file_path.name,
            'file_type': file_path.suffix.lower(),
            'elements': elements,
            'text_elements': [],
            'table_elements': [],
            'metadata': {
                'total_elements': len(elements),
                'loaded_at': datetime.utcnow().isoformat(),
            }
        }
        
        for i, elem in enumerate(elements):
            elem_type = type(elem).__name__
            content = getattr(elem, 'text', str(elem))
            
            if 'Table' in elem_type:
                result['table_elements'].append({
                    'index': i,
                    'type': 'table',
                    'content': content,
                })
            else:
                result['text_elements'].append({
                    'index': i,
                    'type': 'text',
                    'content': content,
                })
        
        result['metadata'].update({
            'text_elements': len(result['text_elements']),
            'table_elements': len(result['table_elements']),
        })
        
        print(f"✓ Loaded {len(elements)} elements")
        return result

loader = SimpleDocumentLoader(api_key=UNSTRUCTURED_API_KEY)
print("✓ DocumentLoader ready")

✓ DocumentLoader ready


### Load Test Document

In [ ]:
# Check for sample documents or create one
test_file = None
sample_paths = [Path('uploads'), Path('.'), Path('../')]

for path in sample_paths:
    if path.exists():
        pdfs = list(path.glob('*.pdf'))
        docxs = list(path.glob('*.docx'))
        if pdfs:
            test_file = pdfs[0]
            break
        if docxs:
            test_file = docxs[0]
            break

if not test_file:
    # Create demo document
    demo_html = Path('demo_document.html')
    demo_html.write_text("""
    <!DOCTYPE html>
    <html><body>
    <h1>RAG Pipeline Test Document</h1>
    <p>This is a demonstration document for testing the RAG pipeline.</p>
    
    <h2>Section 1: Introduction</h2>
    <p>Lorem ipsum dolor sit amet, consectetur adipiscing elit. 
    Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua.</p>
    
    <h2>Section 2: Sales Data</h2>
    <table border="1"><tr><th>Product</th><th>Q1</th><th>Q2</th><th>Q3</th></tr>
    <tr><td>Product A</td><td>1000</td><td>1500</td><td>2000</td></tr>
    <tr><td>Product B</td><td>800</td><td>1200</td><td>1600</td></tr></table>
    
    <h2>Section 3: Analysis</h2>
    <p>Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore.</p>
    </body></html>
    """)
    test_file = demo_html
    print(f"Created demo file: {test_file.name}")
else:
    print(f"Using sample file: {test_file.name}")

In [ ]:
# Load document
doc_data = loader.load(str(test_file))

print(f"\n📄 Document Loaded:")
print(f"  File: {doc_data['file_name']}")
print(f"  Format: {doc_data['file_type']}")
print(f"  Total elements: {doc_data['metadata']['total_elements']}")
print(f"  Text elements: {doc_data['metadata']['text_elements']}")
print(f"  Table elements: {doc_data['metadata']['table_elements']}")

print(f"\nSample content (first 3 elements):")
for elem in doc_data['text_elements'][:3]:
    preview = elem['content'][:80].replace('\n', ' ')
    print(f"  • {preview}...")

## Step 3: Chunking

In [ ]:
class RAGChunker:
    def __init__(self, chunk_size=800, chunk_overlap=150):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
    
    def chunk(self, doc_data: Dict[str, Any]) -> List[Document]:
        chunks = []
        chunk_id = 0
        
        # Tables (preserve as-is)
        for elem in doc_data['table_elements']:
            chunks.append(Document(
                page_content=elem['content'],
                metadata={
                    'source': doc_data['file_name'],
                    'type': 'table',
                    'chunk_id': chunk_id,
                }
            ))
            chunk_id += 1
        
        # Text (split intelligently)
        all_text = '\n\n'.join([e['content'] for e in doc_data['text_elements']])
        text_chunks = self.splitter.split_text(all_text)
        
        for text in text_chunks:
            chunks.append(Document(
                page_content=text,
                metadata={
                    'source': doc_data['file_name'],
                    'type': 'text',
                    'chunk_id': chunk_id,
                }
            ))
            chunk_id += 1
        
        return chunks

chunker = RAGChunker(chunk_size=800, chunk_overlap=150)
chunks = chunker.chunk(doc_data)

print(f"✓ Chunking complete:")
print(f"  Total chunks: {len(chunks)}")
print(f"  Text chunks: {len([c for c in chunks if c.metadata['type'] == 'text'])}")
print(f"  Table chunks: {len([c for c in chunks if c.metadata['type'] == 'table'])}")

## Step 4: Embeddings

In [ ]:
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"✓ Model loaded: {embedding_dim} dimensions")

print(f"\nEmbedding {len(chunks)} chunks...")
texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

# Add to chunks
for chunk, emb in zip(chunks, embeddings):
    chunk.metadata['embedding'] = emb.tolist()

print(f"✓ Embeddings complete")

## Step 5: Vector Storage (pgvector)

In [ ]:
class VectorStore:
    def __init__(self, db_url: str, tenant_id: str):
        self.db_url = db_url
        self.tenant_id = tenant_id
        self.connected = False
        
        if db_url:
            try:
                from sqlalchemy import create_engine, text
                self.engine = create_engine(db_url)
                with self.engine.connect() as conn:
                    conn.execute(text("SELECT 1"))
                    self.connected = True
                print("✓ Database connected")
            except Exception as e:
                print(f"⚠ Database connection failed: {e}")
    
    def store(self, chunks: List[Document]) -> int:
        if not self.connected:
            print("⚠ Database not connected, skipping storage")
            return 0
        
        try:
            from sqlalchemy import text
            stored = 0
            with self.engine.connect() as conn:
                for chunk in chunks:
                    try:
                        embedding = chunk.metadata.get('embedding')
                        if not embedding:
                            continue
                        
                        stmt = text("""
                            INSERT INTO knowledge_documents 
                            (id, tenant_id, document_id, content, embedding, metadata, created_at)
                            VALUES (:id, :tenant_id, :doc_id, :content, :emb, :meta, :created)
                        """)
                        
                        conn.execute(stmt, {
                            'id': str(uuid.uuid4()),
                            'tenant_id': self.tenant_id,
                            'doc_id': str(uuid.uuid4()),
                            'content': chunk.page_content,
                            'emb': str(embedding),
                            'meta': json.dumps(chunk.metadata),
                            'created': datetime.utcnow().isoformat()
                        })
                        stored += 1
                    except:
                        pass
                
                conn.commit()
            
            print(f"✓ Stored {stored} chunks")
            return stored
        except Exception as e:
            print(f"Error storing: {e}")
            return 0

vector_store = VectorStore(DATABASE_URL, "test-tenant")
vector_store.store(chunks)

## Step 6: Retrieval

In [ ]:
class Retriever:
    def __init__(self, embedding_model, chunks: List[Document]):
        self.model = embedding_model
        self.chunks = chunks
        self.embeddings = np.array([np.array(c.metadata['embedding']) for c in chunks])
    
    def search(self, query: str, top_k=3):
        from scipy.spatial.distance import cosine
        
        query_emb = self.model.encode([query])[0]
        similarities = [1 - cosine(query_emb, emb) for emb in self.embeddings]
        
        top_idx = np.argsort(similarities)[::-1][:top_k]
        results = []
        
        for idx in top_idx:
            chunk = self.chunks[idx]
            results.append({
                'content': chunk.page_content,
                'type': chunk.metadata['type'],
                'similarity': similarities[idx],
            })
        
        return results

retriever = Retriever(embedding_model, chunks)
print("✓ Retriever ready")

In [ ]:
# Test retrieval
test_queries = [
    "What is the document about?",
    "Tell me about the sales data",
    "What information is provided?"
]

print("\n🔍 Retrieval Test\n")
for query in test_queries:
    results = retriever.search(query, top_k=2)
    print(f"Q: {query}")
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['type'].upper()}] {r['similarity']:.2%} match")
        print(f"     {r['content'][:60]}...")
    print()

## Step 7: LLM Generation

In [ ]:
llm = None
try:
    if OPENROUTER_API_KEY:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(
            model="openrouter/meta-llama/llama-2-7b-chat",
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1",
            temperature=0.7,
            max_tokens=300
        )
        print("✓ LLM configured (OpenRouter)")
except:
    print("⚠ LLM not available")

In [ ]:
# Full RAG pipeline test
test_query = "What is this document about?"

print(f"\n🔄 FULL RAG PIPELINE\n")
print(f"Query: {test_query}\n")

# Retrieve
print("[1] RETRIEVAL")
retrieved = retriever.search(test_query, top_k=3)
for i, r in enumerate(retrieved, 1):
    print(f"  {i}. [{r['type'].upper()}] {r['similarity']:.1%}")

# Generate
print("\n[2] GENERATION")
if llm:
    context = '\n\n'.join([r['content'] for r in retrieved])
    prompt = f"Based on: {context}\n\nQ: {test_query}\nA:"
    response = llm.invoke(prompt)
    print(f"\nAnswer:\n{response.content}")
else:
    print(f"\nContext Summary:")
    for i, r in enumerate(retrieved, 1):
        print(f"  {i}. {r['content'][:80]}...")

## Summary

In [ ]:
print("""
✅ RAG PIPELINE COMPLETE

📊 Pipeline Summary:
  • Document loaded: yes
  • Elements extracted: text + tables
  • Chunks created: """ + str(len(chunks)) + """
  • Embeddings generated: """ + str(len(chunks)) + """
  • Retrieval tested: yes
  • Generation tested: """ + ("yes" if llm else "demo mode") + """

🚀 Next Steps:
  1. Create UnstructuredDocumentLoader in document_processor.py
  2. Update rag_service.py to use Unstructured
  3. Test with real documents
  4. Deploy to production
""")